# set up
  



In [ ]:
# standard 
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# datasets 
import sklearn.datasets
from sklearn.linear_model import LogisticRegression

# models 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

# plotting specials
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError
import matplotlib.patches as mpatches


# global settings 
seed = 1356



In [ ]:
def plot_decision_boundary(model, model_name, sets_array, *, sets_array_info=None, spec_lim=None):
    '''
    it plots decision boundary based on model. model name will be printed while X, y will be scattered with suitable colors:
    model: give sklearn model with predict method,
    model_name: string which will be displayed
    sets_array: list containg all sets for plotting: ()
    sets_array_info: dictionary with settings for every plot for labels information
    spec_lim: (x_min, x_max, y_min, y_max)
    Plot names: to manual adjust 

    IMPORTANT:
    - shape: sets_array = sets_array_info, array type
    - sets_array - list of lists


    libraries:
    from sklearn.utils.validation import check_is_fitted
    from sklearn.exceptions import NotFittedError
    import matplotlib.patches as mpatches
    '''
    # local variables
    dim = [len(sets_array), len(sets_array[0])]

    # creating settings for grid 
    if spec_lim is None:
        X_list = [sets_array[i][j][0] for i in range(dim[0]) for j in range(dim[1])]
        x1_min = min((X[:, 0].min() for X in X_list)) - 1 
        x1_max = min((X[:, 0].max() for X in X_list)) + 1
        x2_min = min((X[:, 1].min() for X in X_list)) - 1
        x2_max = min((X[:, 1].max() for X in X_list)) + 1
    else:
        x1_min, x1_max, x2_min, x2_max = spec_lim
    h = max((x1_max - x1_min)/x1_max, (x2_max - x2_min)/x2_max) / (100 * 2)

    # creating grid 
    xx1, xx2 = np.meshgrid(np.arange(x1_min, x1_max, h),
                         np.arange(x2_min, x2_max, h))
    grid_points = np.c_[xx1.ravel(), xx2.ravel()]    

    # creating figure with plots
    fig, ax = plt.subplots(nrows=dim[0], ncols=dim[1], figsize=(10, 8), sharex=True, sharey=True)


    # iterating through id
    for id in [(i, j) for i in range(dim[0]) for j in range(dim[1])]:

        # 1 element case
        if dim == [1,1]:
            ax = np.array([[ax]])

        # predicting values on grid
        X, y = sets_array[id[0]][id[1]]
        try:
            check_is_fitted(model)
            is_fitted = True
        except NotFittedError:
            is_fitted = False
        if not is_fitted:
            model.fit(X, y)
        
        if hasattr(model, "predict"):
            Z = model.predict(grid_points).reshape(xx1.shape)
        # TODO if some models conatain not class values but float values need to add treshold here
        else:
            raise ValueError("Model does not have predict method.")


        # plotting
        ax[id].contourf(xx1, xx2, Z, alpha=0.3, cmap='bwr')
        ax[id].contour(xx1, xx2, Z, colors='black', linewidths=0.4) 
        # X points 
        ax[id].scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolor='k', s=20)

        # labels (subplot)
        if sets_array_info is not None:
            title = f"{sets_array_info[id]}"
            ax[id].set_title(title, fontsize=10)

        # classes names 
        # TODO ~ more clean, make sure it really returns right classes
        legend_labels = {i: f'Class {i}' for i in range(len(np.unique(y)))}  # możesz zmieniać
        handles = [
            mpatches.Patch(color=plt.cm.bwr(contour_norm), label=legend_labels[label])
            for label, contour_norm in zip(np.unique(Z), np.linspace(0, 1, len(np.unique(Z))))
        ]
        ax[id].legend(handles=handles, title="Model prediction", fontsize=8, loc='best')
        # ax[id].legend(loc='best', fontsize=8)
        ax[id].grid(True)

    # labels (figure)
    fig.suptitle(f'Decision Boundaries for {model_name}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


# Task 1

Use the function: 

```python
sklearn.datasets.make_moons
```
to generate and plot four sets, where

  * n_samples=1000, noise=.05
  * n_samples=1000, noise=.3
  * n_samples=200, noise=.05
  * n_samples=200, noise=.3



In [ ]:
# list with sets parameters
sets_array_param = np.array((
    (
    dict(n_samples=1000, noise=0.05),
    dict(n_samples=1000, noise=0.3)
    ),
    (
        dict(n_samples=200, noise=0.05),
        dict(n_samples=200, noise=0.3)
    )))

# list with loaded arrays
sets_array = [
    [sklearn.datasets.make_moons(**param, random_state=seed) for param in row]
    for row in sets_array_param]


In [ ]:
# plotting sets
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(8,8), sharex=True, sharey=True)
for id in [(i, j) for i in range(2) for j in range(2)]:
    X, y = sets_array[id[0]][id[1]]
    xx, yy = np.meshgrid(X[:, 0], X[:, 1])
    ax[id].scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolor='k', s=20)


# Task 2 & 3 

## task 2
Use the above created datasets to train an SVM model by function:
```python
SVC(kernel='linear', C=1.0, random_state=0)
```
Illustrate the SVM lines on the plot. 

## Task 3
Repeat Task 2 for models 
```python
svm = SVC(kernel='rbf', random_state=0, gamma=0.1, C=1.0)
```
```python
svm=SVC(kernel='rbf', random_state=0, gamma=0.1, C=100.0)
```
```python
svm=SVC(kernel='rbf', random_state=0, gamma=100, C=10)
```

Which model is the best?



## Solution 

### Goal

Compare how certain hyperparameters influence the model, and identify which models are overfitted or underfitted.

### Theoretical introduction

The objective function minimized by Support Vector Machines is often of the form:

$$
C \sum_{i=1}^n \mathcal{L}(f(x_i), y_i) + \Omega(w)
$$

Where:

- $C$ is a regularization hyperparameter,
- $\mathcal{L}$ is the loss function (e.g. hinge loss),
- $\Omega(w)$ is the regularization term (e.g. $\frac{1}{2} \|w\|^2$).

The parameter $C$ controls the trade-off between maximizing the margin and minimizing classification errors. A large $C$ places a higher penalty on misclassification, so the model tries to fit the data more tightly, resulting in a smaller margin. This may lead to overfitting, as small changes in the input space can lead to different classification decisions.

In contrast, a small $C$ allows the model to tolerate more classification errors, resulting in a wider margin and better generalization, but potentially at the cost of underfitting, especially if relevant support vectors are ignored.

The $\gamma$ hyperparameter appears in RBF (radial basis function) kernels, such as:

$$
K(x, x') = \exp(-\gamma \|x - x'\|^2)
$$

Here, $\gamma$ controls how far the influence of a single training example reaches. High $\gamma$ values cause the influence to be very local (only close points matter), leading to a highly flexible model, which can overfit. Low $\gamma$ values make the influence broader, which smooths the decision boundary and may lead to underfitting.

**Summary:**

- $C$ controls the margin by balancing classification error vs. complexity.
- $\gamma$ controls the shape of the decision boundary by affecting the kernel width.
- They must be tuned together for optimal

### Methodology
- i created plots which are drawn on same length axis
- i do not check any models stats as accuracy f score etc. Point is to have intuition and catch visuallty what is going on.
### Conclusions 
- Linear model & rbf model with $\gamma = 0.1, C=1$:
    - Main difference we can is taht *rbf* model is adjusting to the shape of data, which cause model is better fit of model
- rbf model $\gamma=0.1, C=1.000$
    - We can see difference between *rbf* example with smaller $C$ because here our model is more fitted to data and more narrower margin. 
- model with $\gamma = 100, C=10$
    - I do not know what cause that almost all predictions are blue, $C$ will cause narrow margin and $\gamma$ will cause impact only of local points but i do not understand how this impact almost all clasifications are blue  



In [ ]:
model_list = [
    (SVC(kernel='linear', C=1.0, random_state=seed), dict(kernel='linear')),
    (SVC(kernel='rbf', random_state=seed, gamma=0.1, C=1.0), dict(kernel='rbf', gamma=0.1, C=1.0)),
    (SVC(kernel='rbf', random_state=seed, gamma=0.1, C=100.0), dict(kernel='rbf', gamma=0.1, C=100.0)),
    (SVC(kernel='rbf', random_state=seed, gamma=100, C=10), dict(kernel='rbf', gamma=100, C=10))
]

for model, model_parameters in model_list:
    # creating plot string
    if len(model_parameters) > 1:
        model_name = f'SVC {model_parameters['kernel']} model with $\gamma$={model_parameters['gamma']}, $C$={model_parameters['C']}'
    else:
        model_name = f'SVC {model_parameters['kernel']} model'

    # plotting 
    plot_decision_boundary(model, model_name, sets_array, sets_array_info=sets_array_param)


# Task 4 & 5

## Task 4
Use cross-validation to find the best parameters for a logistic model trained on Iris dataset.
Illustrate this dataset(the same two features as in Task 4) and prediction of the model. 

Compare this model with the model form Task 4.

# Task 5
# Task 4 
Use cross-validation to find the best parameters for an SVM model trained on Iris dataset.
Before fitting the model, rescale data
```python
from sklearn.preprocessing import StandardScaler
```

Illustrate this dataset(use two features) and prediction of the model. 


### Goal
We want to show example where SVM is similiar to Logistic Regression due to fact. 
### Methods
We create two pipelines, one containing SVC model and the other containing Logistic Regression Model and than compare it
### Solutions
In region with most data we can see that model behaves similiar, to each other, differences occure on larger distances where amount of points are reduce and shape boundary is more visible (stright line vs nonlinear type). 

In [ ]:
# loading iris data set
iris = sklearn.datasets.load_iris()
X = iris.data[:, :2]  
y = iris.target

# creating model pipeline 
model_array = [
    (make_pipeline(
        StandardScaler(),
        SVC(kernel='rbf')), 'svc'),
    (make_pipeline(
        StandardScaler(),
        LogisticRegression()
    ), 'logisticregression')
]
# Grid search optimization parameters
param_list = [
    dict( # dict for SVC model
        svc__C = np.linspace(0.1, 10, 10, True), 
        svc__gamma = np.linspace(0.01, 1, 20)),
    dict( # dict for LogisticRegression model
        logisticregression__C = np.linspace(0.1, 10, 10, True)
        )
    ]
# splitting data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    # duże C overfitting, małe c underfitting (bo to są kary do przewidywań )
    # TUTAJ KOMENTARZ: bierzemy tylko parametr C ze względu na to że inne typu degree i gamma, obowiązują tylko przy części modeli + 
    # gamma ~ lokalność oddziaływania 
    # druga wersja bierzemy dwa parametry ponieważ jeden wpływa na to ile punktów bierzemy pod uwagę, drugi bierze pod uwagę, jak bardzo każemy złe przyporzadkowanie


for (model, name), params in zip(model_array, param_list):
    # finding best parameters
    grid_search = GridSearchCV(model, params)
    grid_search.fit(X_train, y_train)
    model = grid_search.best_estimator_ # it retuns best trained model
    model_param = grid_search.best_params_ 
    # report information
    print(classification_report(y_test, model.predict(X_test)))
    plot_decision_boundary(model, f'{name} model with $C$={model_param[f'{name}__C']}', [[(X, y)]])

## Task 6
Repeat the task for the data obtained with the yfinance module.


Napisać tutaj że ponieważ mój model z yfinance z poprzedniego zadania nie jest bardzo przemyślany powtarzanie tego modelu za pomocą SVC nie ma sensu ponieważ nie bedzie widać miarodajnych różnic i czego można by się spodziewać. 
 